# Natural Sampling publication pipeline

This notebook is the isolated Natural Sampling copy. The original official pipeline remains unchanged. 
Training is disabled because the selected Phase 1 and Phase 2 models are already archived under `Natural_Sampling/Models`. 
Run the notebook from the first cell to regenerate figures and tables under `Natural_Sampling/Results`.


# B4 C15 — Phase 1 canonique dans trois écosystèmes marocains

Ce notebook constitue l’unique point d’entrée pour reproduire la **Phase 1 supervisée** du modèle B4 dans trois contextes forestiers contrastés : la forêt dense d’Ifran, la forêt de faible densité de Maamoura et la forêt clairsemée d’Agadir.

L’objectif scientifique est d’estimer la hauteur de canopée GEDI RH95 à partir d’un tenseur multimodal strictement limité à **15 canaux natifs**. Aucun canal n’est supprimé à l’exécution : les fichiers NPY stockent directement les variables utilisées par le modèle. Le protocole sépare spatialement TRAIN, VAL et TEST. Le TEST n’intervient ni dans l’optimisation ni dans l’arrêt anticipé.

Le notebook est volontairement linéaire : configuration, contrôle des données, entraînement, sélection sur VAL, puis évaluation finale sur TEST.


## 1 — Hypothèse, architecture et fonction de coût

La Phase 1 apprend une relation spatiale entre les observations satellitaires et GEDI RH95. Le modèle reçoit huit bandes Sentinel‑2, quatre canaux Sentinel‑1, le masque de zone d’étude, le DEM et la pente. Les têtes ordinales et de classification sont désactivées : la sortie est une régression continue.

La fonction supervisée est la perte de Huber :

\[
\mathcal L_{\delta}(r)=
\begin{cases}
\frac{1}{2}r^2, & |r|\leq\delta,\\
\delta\left(|r|-\frac{1}{2}\delta\right), & |r|>\delta,
\end{cases}
\qquad r=\widehat H-H.
\]

Elle conserve une réponse quadratique pour les erreurs modérées, tout en réduisant l’influence des observations extrêmes. Le delta est fixé avant l’évaluation finale : 3 m à Ifran et Agadir, 1 m à Maamoura. L’échantillonnage d’entraînement est équilibré selon les classes de hauteur, mais chaque observation conserve un poids unitaire dans la perte.


In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
CONFIG_PATH = PROJECT / "Config" / "pipeline_config.json"
SOURCE_ROOT = PROJECT / "Source" / "Project"
assert PROJECT.is_dir(), PROJECT
assert CONFIG_PATH.is_file(), CONFIG_PATH
PIPELINE = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("Projet :", PROJECT)
print("Configuration :", CONFIG_PATH)
print("Python :", sys.executable)


## 2 — Contrat d’entrée B4 C15

| Groupe | Canaux |
|---|---|
| Sentinel‑2 | B02, B03, B04, B08, B05, B06, B07, B8A |
| Sentinel‑1 | ASC‑VV, ASC‑VH, DESC‑VV, DESC‑VH |
| Variables auxiliaires | AOI mask, DEM, slope |

Les tenseurs ont la forme `(512, 512, 15)`. PALSAR n’entre pas dans B4 C15. L’ordre est figé par `experiment.json` et vérifié avant tout lancement.


In [ ]:
EXPECTED_CHANNELS = [
    "S2_LEAFON_B02", "S2_LEAFON_B03", "S2_LEAFON_B04", "S2_LEAFON_B08",
    "S2_LEAFON_B05", "S2_LEAFON_B06", "S2_LEAFON_B07", "S2_LEAFON_B8A",
    "S1_ASC_VV", "S1_ASC_VH", "S1_DESC_VV", "S1_DESC_VH", "AOI_MASK", "DEM", "SLOPE",
]

preflight_rows = []
for site, cfg in PIPELINE["sites"].items():
    catalog = Path(cfg["catalog"])
    experiment = json.loads((catalog / "experiment.json").read_text(encoding="utf-8"))
    order = experiment.get("channel_order") or experiment.get("schema", {}).get("channel_order")
    arrays = sorted(catalog.rglob("*.npy"))
    if not arrays:
        raise FileNotFoundError(f"{site}: aucun NPY dans {catalog}")
    shape = tuple(np.load(arrays[0], mmap_mode="r", allow_pickle=False).shape)
    required = [catalog / "sample_catalog.csv", catalog / "shot_catalog.csv.gz", catalog / "spatial_split_manifest.csv"]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(missing)
    if shape != (512, 512, 15):
        raise RuntimeError((site, shape, arrays[0]))
    if list(order) != EXPECTED_CHANNELS:
        raise RuntimeError((site, order))
    preflight_rows.append({"forest": cfg["forest"], "catalog": str(catalog), "shape": shape, "channels": len(order), "status": "PASS"})

pd.DataFrame(preflight_rows)


## 3 — Séparation spatiale et domaines d’évaluation

Les partitions sont figées dans `spatial_split_manifest.csv`. Une même unité spatiale ne doit jamais apparaître dans plusieurs partitions. TRAIN sert à ajuster les poids, VAL contrôle l’arrêt anticipé et les checkpoints, et TEST reste réservé au reporting final.

| Forêt | Domaine d’entraînement | Domaine primaire VAL/TEST |
|---|---:|---:|
| Ifran | 0–45 m | 2–45 m |
| Maamoura | 0–20 m | 2–20 m |
| Agadir | 0–20 m | 2–20 m |

La borne de 2 m définit seulement le domaine d’évaluation : les observations plus basses restent disponibles pendant l’apprentissage afin de représenter le sol et les couverts très bas.


In [ ]:
split_rows = []
for site, cfg in PIPELINE["sites"].items():
    manifest = pd.read_csv(Path(cfg["catalog"]) / "spatial_split_manifest.csv")
    split_col = next(name for name in ("split", "subset", "partition") if name in manifest.columns)
    id_col = next(name for name in ("spatial_group_id", "spatial_id", "patch_id", "tile_id", "group_id") if name in manifest.columns)
    memberships = manifest.groupby(id_col)[split_col].nunique()
    leaks = int((memberships > 1).sum())
    if leaks:
        raise RuntimeError(f"{site}: {leaks} unités spatiales traversent plusieurs splits")
    counts = manifest[split_col].value_counts().to_dict()
    split_rows.append({"forest": cfg["forest"], "spatial_units": manifest[id_col].nunique(), "leaks": leaks, **counts})
pd.DataFrame(split_rows)


## 4 — Paramètres d’optimisation communs et spécifiques

Les trois modèles utilisent l’architecture Hytec B4, AdamW, un learning rate initial de `1e-4`, un weight decay de `5e-3`, un gradient clipping de 1, un batch de 8, un dropout de 0,15 et la seed 42. La validation intervient tous les 66 steps et la patience est de 15 évaluations.

La différence de delta n’est pas un changement d’architecture : elle règle uniquement la transition quadratique‑linéaire de Huber en fonction de la distribution des résidus observée sur VAL.


In [ ]:
phase1_protocol = []
for site, cfg in PIPELINE["sites"].items():
    phase1_protocol.append({
        "forest": cfg["forest"], "catalog": cfg["catalog"], "huber_delta": cfg["huber_delta"],
        "validation_domain_m": cfg["primary_validation_domain_m"], "test_domain_m": cfg["primary_test_domain_m"],
        "batch_size": PIPELINE["protocol"]["batch_size"], "seed": PIPELINE["protocol"]["seed"],
    })
pd.DataFrame(phase1_protocol)


## 5 — Entraînement Phase 1 depuis zéro

Les interrupteurs sont désactivés par défaut afin d’éviter un relancement accidentel. Pour reproduire une forêt, passer uniquement son indicateur à `True`. Le lanceur protège les runs terminés, reprend un run interrompu seulement si un checkpoint `last` existe, et refuse d’écraser un dossier non reprenable.

Lorsque le run se termine, le checkpoint choisi sur VAL est automatiquement copié vers le chemin canonique attendu par la Phase 2 et un fichier `phase1_final_model.json` enregistre son SHA‑256. Ainsi, aucun modèle Phase 1 préexistant n’est nécessaire : il est produit par cette cellule.


In [ ]:
# Interrupteur unique de reproduction.
# False = audit/consultation sans entraînement; True = entraîne les trois forêts.
REPRODUCE_PHASE1_FROM_SCRATCH = False
RUN_PHASE1 = {site: REPRODUCE_PHASE1_FROM_SCRATCH for site in ("ifran", "maamoura", "agadir")}

TRAIN_SCRIPT = SOURCE_ROOT / "run_b4_c15_train.py"
assert TRAIN_SCRIPT.is_file(), TRAIN_SCRIPT
print(
    "[MODE PHASE 1] "
    + ("REPRODUCTION ACTIVE — entraînement réel des trois forêts" if REPRODUCE_PHASE1_FROM_SCRATCH
       else "AUDIT SEULEMENT — aucun entraînement; passer REPRODUCE_PHASE1_FROM_SCRATCH=True pour reproduire"),
    flush=True,
)

for site, enabled in RUN_PHASE1.items():
    command = [sys.executable, "-u", str(TRAIN_SCRIPT), "--site", site]
    if enabled:
        command.append("--execute")
    print("\n", subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, check=True)


## 6 — Sélection des checkpoints et évaluation finale

Les checkpoints sont créés à partir de VAL. Pour l’article, `best_slope` est utilisé à Ifran et Agadir, tandis que `best_any` est utilisé à Maamoura. Le TEST est évalué en coordonnées GEDI originales avec une seule prédiction par tir (`unique-nearest`). Cette unité évite de surpondérer les tirs présents dans plusieurs fenêtres ou patches.

STEP07 lit le run nouvellement créé sous `Runs/`, évalue explicitement le checkpoint retenu, puis publie les prédictions dans `Results/Final_Article/Phase1/<forêt>`. Les scatter plots ne dépendent donc plus d’un cache historique.


In [ ]:
# L'évaluation TEST suit automatiquement le même mode que l'entraînement.
RUN_PHASE1_EVALUATION = REPRODUCE_PHASE1_FROM_SCRATCH
EVAL_SCRIPT = SOURCE_ROOT / "step07_evaluate_and_plot.py"
for site in PIPELINE["sites"]:
    command = [sys.executable, "-u", str(EVAL_SCRIPT), "--site", site, "--split", "test"]
    if RUN_PHASE1_EVALUATION:
        command.append("--execute-if-missing")
        subprocess.run(command, check=True)
    else:
        print("[EVAL DRY-RUN]", subprocess.list2cmdline(command))


## 7 — Scatter plots Phase 1 sur TEST

Les axes commencent à zéro pour conserver une lecture physique cohérente. Le filtre RH95 ≥ 2 m agit sur les observations évaluées, sans déplacer l’origine graphique.


In [ ]:
def _column(frame: pd.DataFrame, candidates: tuple[str, ...]) -> str:
    for name in candidates:
        if name in frame.columns:
            return name
    raise KeyError(f"Aucune colonne parmi {candidates}; colonnes={list(frame.columns)}")

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    residual = y_pred - y_true
    slope, intercept = np.polyfit(y_true, y_pred, 1)
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    r2 = float(1.0 - np.sum(residual ** 2) / np.sum((y_true - y_true.mean()) ** 2))
    std_ratio = float(np.std(y_pred) / np.std(y_true))
    beta = float(y_pred.mean() / y_true.mean())
    kge = float(1.0 - np.sqrt((corr - 1.0) ** 2 + (std_ratio - 1.0) ** 2 + (beta - 1.0) ** 2))
    return {
        "n": len(y_true), "r2": r2, "mae": float(np.mean(np.abs(residual))),
        "rmse": rmse, "bias": float(residual.mean()), "slope": float(slope),
        "intercept": float(intercept), "corr": float(corr),
        "std_ratio": std_ratio, "kge": kge,
    }

def load_prediction_table(path: Path, min_height: float, max_height: float) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    frame = pd.read_csv(path)
    if "aux_shot_uid" in frame.columns:
        order = [name for name in ("abs_temporal_delta_days", "sequence_center_distance", "batch_index", "candidate_order") if name in frame.columns]
        frame = frame.sort_values(order, kind="mergesort") if order else frame
        frame = frame.drop_duplicates("aux_shot_uid", keep="first").copy()
    true_col = _column(frame, ("y_true", "gedi_rh95", "rh95", "target", "observed"))
    pred_col = _column(frame, ("y_pred", "pred_on_growthloss", "prediction_original_coords", "prediction", "predicted"))
    y_true = frame[true_col].to_numpy(float)
    y_pred = frame[pred_col].to_numpy(float)
    keep = np.isfinite(y_true) & np.isfinite(y_pred) & (y_true >= min_height) & (y_true <= max_height)
    return frame.loc[keep].copy(), y_true[keep], y_pred[keep]

def scatter_panel(path: Path, forest: str, domain: tuple[float, float], output: Path) -> dict:
    _, y_true, y_pred = load_prediction_table(path, *domain)
    metrics = regression_metrics(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7.2, 6.4), constrained_layout=True)
    hb = ax.hexbin(y_true, y_pred, gridsize=48, mincnt=1, bins="log", cmap="viridis")
    low, high = 0.0, float(domain[1])
    ax.plot([low, high], [low, high], "k--", lw=1.5, label="1:1")
    x = np.linspace(low, high, 100)
    ax.plot(x, metrics["intercept"] + metrics["slope"] * x, color="crimson", lw=1.8, label="Régression")
    ax.set(xlim=(low, high), ylim=(low, high), xlabel="GEDI RH95 observé (m)", ylabel="Hauteur prédite (m)")
    ax.set_title(f"{forest} — TEST unique-nearest — RH95 {domain[0]:g}–{domain[1]:g} m", fontweight="bold")
    text = "\n".join([
        f"n = {metrics['n']}", f"R² = {metrics['r2']:.4f}", f"MAE = {metrics['mae']:.4f} m",
        f"RMSE = {metrics['rmse']:.4f} m", f"Slope = {metrics['slope']:.4f}",
        f"Std ratio = {metrics['std_ratio']:.4f}", f"Bias = {metrics['bias']:+.4f} m",
        f"KGE = {metrics['kge']:.4f}",
    ])
    ax.text(0.025, 0.975, text, transform=ax.transAxes, va="top", bbox=dict(facecolor="white", alpha=.9))
    ax.legend(loc="lower right")
    fig.colorbar(hb, ax=ax, label="log10(N)")
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=600, bbox_inches="tight")
    fig.savefig(output.with_suffix(".pdf"), bbox_inches="tight")
    plt.show()
    return metrics


In [ ]:
phase1_metrics = []
for site, cfg in PIPELINE["sites"].items():
    path = Path(cfg["test_predictions"])
    if not path.is_file():
        print("[MISSING — lancer STEP07]", site, path)
        continue
    domain = tuple(map(float, cfg["primary_test_domain_m"]))
    output = PROJECT / "Results" / "Final_Article" / "Phase1" / cfg["forest"] / "scatter_test.png"
    metrics = scatter_panel(path, cfg["forest"], domain, output)
    phase1_metrics.append({"forest": cfg["forest"], **metrics})
phase1_metrics = pd.DataFrame(phase1_metrics)
phase1_metrics


## 8 — Interprétation pour l’article

La Phase 1 fournit une estimation spatialement supervisée et constitue le parent immuable de la Phase 2. Les écarts de performance entre écosystèmes doivent être interprétés avec leur distribution de hauteurs, leur structure spatiale et leur signal satellitaire. Le R² plus faible d’Agadir n’implique pas automatiquement une erreur absolue plus forte : dans une forêt basse et peu variable, une faible variance de RH95 réduit mécaniquement le R².

Les limites principales sont l’incertitude GEDI, la saturation optique/radar dans les canopées hautes, le décalage temporel entre acquisitions et la représentativité spatiale des tirs GEDI.
